Spam Detection

In [1]:
import numpy as np
import pandas as pd

In [10]:
# load dataset
df = pd.read_csv("spam.csv", encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [12]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [13]:
df = df[['v1', 'v2']]

In [14]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [15]:
class SVM:
    def __init__(self, learning_rate=0.001, C=1.0, epochs=1000):
        self.lr = learning_rate
        self.C = C
        self.epochs = epochs
        self.weights = None
        self.bias = None
    
    # train the model
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # convert labels to -1 and 1 (SVM uses -1/1 not 0/1)
        y_ = np.where(y <= 0, -1, 1)
        
        # initialize weights and bias
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # gradient descent
        for _ in range(self.epochs):
            for idx, x_i in enumerate(X):
                
                # check if point is correctly classified with margin
                condition = y_[idx] * (np.dot(x_i, self.weights) + self.bias) >= 1
                
                if condition:
                    # point is outside margin — only regularization update
                    self.weights = self.weights - self.lr * (2 * self.weights)
                    
                else:
                    # point is inside or wrong side of margin — full update
                    self.weights = self.weights - self.lr * (2 * self.weights - self.C * np.dot(x_i, y_[idx]))
                    self.bias    = self.bias    + self.lr * self.C * y_[idx]
    
    # predict
    def predict(self, X):
        z = np.dot(X, self.weights) + self.bias
        return np.sign(z)  # returns -1 or 1


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split                                                                                                                                                              
from sklearn.preprocessing import StandardScaler                                                                                                           
# convert labels: ham=0, spam=1
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})

# convert text to numbers using TF-IDF
tfidf = TfidfVectorizer(max_features=500)
X = tfidf.fit_transform(df['v2']).toarray()
y = df['v1'].values

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train
model = SVM(learning_rate=0.001, C=1.0, epochs=100)
model.fit(X_train, y_train)

# predict
predictions = model.predict(X_test)
predictions = np.where(predictions == -1, 0, 1)

# accuracy
accuracy = np.mean(predictions == y_test)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8655
